$$ Resume $$ $$ Screening System $$ $$version 2 $$

In [1]:
import numpy as np 
import pandas as pd 
import re
import nltk
import string
from nltk.corpus import stopwords

In [2]:
skills = [
    # Programming Languages
    'python', 'java', 'c language', 'c++', 'c#', 'javascript', 'typescript',
    'php', 'ruby', 'go language', 'rust', 'kotlin', 'swift', 'r language', 'scala',

    # Databases
    'sql', 'mysql', 'postgresql', 'mongodb', 'sqlite', 'oracle',
    'redis', 'cassandra', 'firebase',

    # Data Science & ML
    'machine learning', 'deep learning', 'artificial intelligence',
    'data science', 'data analysis', 'data mining',
    'tensorflow', 'keras', 'pytorch', 'scikit-learn',
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'opencv', 'nltk', 'spacy', 'xgboost', 'lightgbm',

    # Web Development
    'html', 'css', 'bootstrap', 'tailwind',
    'react', 'nextjs', 'vue', 'angular',
    'nodejs', 'express', 'django', 'flask',
    'fastapi', 'spring boot', 'laravel',

    # Cloud & DevOps
    'aws', 'azure', 'gcp',
    'docker', 'kubernetes', 'jenkins',
    'terraform', 'ansible',
    'ci/cd', 'github actions',

    # Version Control
    'git', 'github', 'gitlab', 'bitbucket',

    # Big Data
    'hadoop', 'spark', 'kafka', 'hive',

    # Operating Systems
    'linux', 'unix', 'windows',

    # APIs
    'rest api', 'graphql', 'microservices',

    # Mobile Development
    'android', 'ios', 'flutter', 'react native',

    # Testing
    'selenium', 'pytest', 'junit', 'postman',

    # BI Tools
    'power bi', 'tableau', 'excel',

    # CS Fundamentals
    'data structures', 'algorithms',
    'object oriented programming',
    'oop', 'operating systems',
    'computer networks', 'dbms'
]

In [3]:
def extract_skills(text):
    text = text.lower()
    found = []
    for skill in skills :
        if skill in text:
            found.append(skill)
    return found

In [4]:
df = pd.read_csv(r'C:\Users\murli mishra\OneDrive\Desktop\PYTHON\ResumeScreeningSystem\ResumeV2\postings.csv')

In [ ]:
df.head()
# df['description'] = df['description'].fillna('')

In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+www\S+','',text)
    text = re.sub(r'\S+@+\S+','',text)
    text = re.sub(r'\d+','',text)
    text = text.translate(str.maketrans('','',string.punctuation))
    text = re.sub(r'\s+',' ',text)
    return text

In [6]:
df['description'] = df['description'].fillna('')

In [7]:
df['cleaned'] = df['description'].apply(clean_text)


In [ ]:
# df['cleaned'].iloc[0][:1000]
df.head()

In [ ]:
# total job roles
df['title'].nunique()

In [8]:
# now removing stopwords from the cleaned text

stop = set(stopwords.words('english'))
stop.discard('not')
stop.discard('without')
stop.discard('with')
stop.discard('no')


df['cleaned'] = df['cleaned'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop]))

In [18]:
X = df['cleaned']
y = df['title']

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
tf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)
# df['encoded_title'] = le.fit_transform(df['title'])




In [ ]:
df.head()

In [19]:
X = tf.fit_transform(X)


In [12]:
import pickle
from scipy.sparse import save_npz
from scipy.sparse import load_npz

In [ ]:
df.to_csv('jobs.csv', index=False)

In [21]:
# tf = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))
save_npz('job_matrix.npz', X)
# jobs = pd.read_csv('jobs.csv')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_jobs(resume_text):
    cleaned_resume = clean_text(resume_text)
    cleaned_resume = ' '.join(
        word
        for word in cleaned_resume.split()
        if word not in stop
    )
    resume_vector = tf.transform([cleaned_resume])
    similarities = cosine_similarity(
        resume_vector,
        X
    )
    indices = similarities[0].argsort()[::-1][:10]

    result = jobs.iloc[
    indices
    ][['title', 'description', 'skills']].copy()

    result['score'] = (
    similarities[0][indices] * 100
).round(2)

    return result , indices




In [ ]:
jobs.head()

In [ ]:
recommend_jobs(jobs.iloc[0]['description'])

In [ ]:
jobs.columns


In [ ]:
skills = extract_skills(jobs.iloc[1000]['description'])


In [ ]:
skills

In [ ]:
jobs.iloc[0]['skills_desc']

In [ ]:
jobs['corpus'] = (
    jobs['title'].fillna('') + ' ' + jobs['description'].fillna('') + ' ' + jobs['skills_desc'].fillna('')
)

In [ ]:
X = tf.fit_transform(jobs['corpus'])

In [ ]:
save_npz('job_matrix.npz', X)

In [ ]:
jobs['skills'] = jobs['corpus'].apply(extract_skills)

In [ ]:
jobs.head()

In [ ]:
jobs.head()

In [ ]:
jobs = pd.read_csv('jobs.csv')

In [ ]:
jobs.head()

In [ ]:
def extract_text_from_pdf(pdf_path):
    from PyPDF2 import PdfReader
    reader = PdfReader(pdf_path)
    text = ''
    for page in reader.pages:
        text += page.extract_text() + ' '
    return text.strip()

In [ ]:
resume_text = extract_text_from_pdf(r'C:\Users\murli mishra\OneDrive\Desktop\PYTHON\ResumeV2\Murlidhar_Mishra_Resume_FINAL.pdf')

In [ ]:
recommend_jobs(resume_text)

In [ ]:
print(len(jobs))

In [ ]:
jobs = jobs.reset_index(drop=True)
tf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=2)
X = tf.fit_transform(jobs['corpus'])

In [ ]:
save_npz('job_matrix.npz', X)

pickle.dump(
    tf,
    open('tfidf_vectorizer.pkl', 'wb')
)

jobs.to_csv(
    'jobs.csv',
    index=False
)


In [ ]:
jobs = pd.read_csv('jobs.csv')

In [ ]:
X.shape[0]

In [ ]:
jobs.head()

In [ ]:
recommendations, indices = recommend_jobs(resume_text)
top_idx = indices[0]
job_skills = jobs.iloc[top_idx]['skills']

In [ ]:
resume_skills = extract_skills(resume_text)
resume_skills

In [ ]:
recommendations

In [ ]:
job_skills


In [ ]:
import ast
jobs['skills'] = jobs['skills'].apply(ast.literal_eval)

In [ ]:
missing_skills = list(set(job_skills) - set(resume_skills))
missing_skills

In [ ]:
print("Resume Skills:", resume_skills)
print("Job Skills:", job_skills)

In [ ]:
jobs['skills'] = jobs['corpus'].apply(extract_skills)

resume_skills = extract_skills(resume_text)

In [ ]:
roadmap = {
    'docker': 'Learn Docker and containerization.',
    'aws': 'Learn AWS (EC2, S3, IAM).',
    'kubernetes': 'Learn Kubernetes basics and deployments.',
    'react': 'Build frontend projects with React.',
    'sql': 'Practice SQL queries, joins and window functions.',
    'tensorflow': 'Build deep learning projects using TensorFlow.',
    'pytorch': 'Learn PyTorch and implement neural networks.',
    'git': 'Learn Git workflows and GitHub collaboration.',
    'linux': 'Learn Linux commands and shell scripting.',
    'mongodb': 'Learn NoSQL databases and MongoDB.'
}

def ats_analysis(resume_text):

    recommendations, indices = recommend_jobs(resume_text)

    top_idx = indices[0]

    top_job = jobs.iloc[top_idx]

    job_skills = top_job['skills']
    resume_skills = extract_skills(resume_text)

    matched_skills = list(
        set(job_skills) & set(resume_skills)
    )

    missing_skills = list(
        set(job_skills) - set(resume_skills)
    )

    skill_score = round(
        len(matched_skills) /
        max(len(job_skills), 1) * 100,
        2
    )

    # Similarity score from recommender
    similarity_score = round(
        recommendations.iloc[0]['score'],
        2
    )

    # Final ATS score
    ats_score = round(
        0.7 * similarity_score +
        0.3 * skill_score,
        2
    )

    learning_recommendations = []

    for skill in missing_skills:
        if skill in roadmap:
            learning_recommendations.append(
                roadmap[skill]
            )

    return {
        'recommended_role': top_job['title'], 'company': top_job['company_name'], 'ats_score': ats_score,'similarity_score': similarity_score,'skill_match_score':skill_score,
       'resume_skills': resume_skills,'job_skills':job_skills,'matched_skills':matched_skills,'missing_skills': missing_skills,'learning_recommendations': learning_recommendations,

        'top_recommendations':
            recommendations
    }

In [ ]:
ats_analysis(resume_text)

In [ ]:
jobs = pd.read_csv('jobs.csv')

In [ ]:
columns_to_keep = [
    'title',
    'company_name',
    'description',
    'location',
    'formatted_work_type',
    'skills',
    'cleaned'
]

jobs = jobs[columns_to_keep]

jobs.to_csv(
    "jobs_small.csv",
    index=False
)

In [23]:
import os

files = [
    "jobs_deploy.csv",
    "job_matrix.npz",
    "tfidf_vectorizer.pkl"
]

for f in files:
    print(f"{f}: {os.path.getsize(f)/1024/1024:.2f} MB")

jobs_deploy.csv: 6.00 MB
job_matrix.npz: 216.48 MB
tfidf_vectorizer.pkl: 0.38 MB


In [ ]:
jobs['description'].str.len().describe()


In [24]:
from scipy import sparse

sparse.save_npz(
    "job_matrix.npz",
    X,
    compressed=True
)

In [ ]:
jobs_deploy = jobs[
    [
        'title',
        'company_name',
        'location',
        'formatted_work_type',
        'skills',
        'description'
    ]
]
jobs_deploy['description'] = jobs_deploy['description'][:300].fillna('')

jobs_deploy.to_csv("jobs_deploy.csv", index=False)